# 9장 로지스틱 회귀와 소프트맥스 회귀 — 시그모이드, 교차 엔트로피, 결정 경계, 다중 분류
『머신러닝과 딥러닝 기초』 실습 노트북

다음을 해 봅니다.
1. 0과 1에 직선을 맞추면 생기는 문제 보기, 시그모이드로 확률 만들기
2. 교차 엔트로피(로그 손실) 계산하기
3. 넘파이 경사 하강법으로 로지스틱 회귀를 학습하고 `LogisticRegression`과 비교하기
4. 특징 두 개의 결정 경계, 특징 30개 모델의 임곗값
5. 펭귄 세 종의 소프트맥스 회귀, 서포트 벡터 머신 맛보기

위에서부터 차례로 실행하세요.

## 1. 직선으로 분류하면
4장의 유방암 데이터에서 세포핵 반지름의 평균(`mean radius`) 하나로 0(양성 혹)과 1(악성)에 직선을 맞춥니다. 예측값이 0보다 작거나 1보다 커져 확률로 읽을 수 없습니다.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression

cancer = load_breast_cancer(as_frame=True)   # 4장의 유방암 데이터
X = cancer.data
y = (cancer.target == 0).astype(int)         # 악성이면 1, 아니면 0
Xtr, Xte, ytr, yte = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

r_tr = Xtr[["mean radius"]]                  # 세포핵 반지름의 평균
r_te = Xte[["mean radius"]]
lin = LinearRegression().fit(r_tr, ytr)      # 0과 1에 직선을 맞춘다
new = pd.DataFrame({"mean radius": [8, 14, 20, 28]})
print(lin.predict(new).round(2))

## 2. 시그모이드 함수
$\sigma(z) = 1/(1 + e^{-z})$는 어떤 $z$를 넣어도 0과 1 사이의 값을 냅니다.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

for z in [-4, -2, 0, 2, 4]:
    print(f"σ({z:2d}) = {sigmoid(z):.3f}")

In [ ]:
import matplotlib.pyplot as plt

zz = np.linspace(-7, 7, 200)
plt.plot(zz, sigmoid(zz), label="sigmoid")
plt.axhline(0.5, ls="--", c="gray"); plt.axvline(0, ls="--", c="gray")
plt.xlabel("z"); plt.ylabel("sigma(z)"); plt.legend()
plt.show()

로지스틱 회귀 $\hat{y} = \sigma(wx + b)$를 학습합니다. `C=np.inf`는 규제(13장)를 끄는 설정입니다. 확률 0.5인 경계는 $z = 0$, 곧 반지름 $-b/w$입니다.

In [ ]:
logi = LogisticRegression(C=np.inf)          # 규제를 끈다(본문 참고)
logi.fit(r_tr, ytr)
w, b = logi.coef_[0, 0], logi.intercept_[0]
print(f"w = {w:.4f}, b = {b:.4f}, 경계 = {-b / w:.2f}")
new = pd.DataFrame({"mean radius": [10, 14, 16, 20]})
print(logi.predict_proba(new)[:, 1].round(3))
print(logi.predict(new))
print(f"테스트 정확도: {logi.score(r_te, yte):.1%}")

## 3. 교차 엔트로피
모델의 로그 손실과, 특징을 보지 않고 악성 비율만 답할 때의 로그 손실(= 레이블의 엔트로피, 자연로그)을 비교합니다.

In [ ]:
from sklearn.metrics import log_loss

p_tr = logi.predict_proba(r_tr)[:, 1]        # 학습 데이터의 악성 확률
base = np.full(len(ytr), ytr.mean())         # 모두에게 악성 비율을 답한다
print(f"악성 비율: {ytr.mean():.3f}")
print(f"모델의 로그 손실:    {log_loss(ytr, p_tr):.4f}")
print(f"비율만 답할 때:      {log_loss(ytr, base):.4f}")

## 4. 경사 하강법으로 학습하기
기울기는 $\frac{1}{n}\sum(\hat{y}_i - y_i)x_i$와 $\frac{1}{n}\sum(\hat{y}_i - y_i)$입니다. 반지름을 표준화하고 학습률 1.0으로 1,000번 업데이트합니다.

In [ ]:
xr = Xtr["mean radius"].to_numpy()
yt = ytr.to_numpy()                          # 레이블(0 또는 1)
m, s = xr.mean(), xr.std()
xs = (xr - m) / s                            # 표준화(8장)

def cross_entropy(w, b):
    p = sigmoid(w * xs + b)
    return -np.mean(yt * np.log(p) + (1 - yt) * np.log(1 - p))

w, b, lr = 0.0, 0.0, 1.0
for step in range(1, 1001):
    p = sigmoid(w * xs + b)                  # 악성일 확률
    gw = np.mean((p - yt) * xs)              # w에 대한 편미분
    gb = np.mean(p - yt)                     # b에 대한 편미분
    w, b = w - lr * gw, b - lr * gb
    if step in [1, 10, 100, 1000]:
        print(f"{step:4d}번: w = {w:.4f}, b = {b:.4f}, "
              f"손실 {cross_entropy(w, b):.4f}")

사이킷런과 비교하고 원래 단위로 되돌립니다. 마지막 줄은 규제를 넣는 기본 설정(`C=1.0`)의 결과입니다.

In [ ]:
sk = LogisticRegression(C=np.inf).fit(xs.reshape(-1, 1), yt)
print(f"경사 하강법       : w = {w:.2f}, b = {b:.2f}")
print(f"LogisticRegression: w = {sk.coef_[0, 0]:.2f}, "
      f"b = {sk.intercept_[0]:.2f}")
print(f"원래 단위로: w = {w / s:.2f}, b = {b - w * m / s:.2f}")
d = LogisticRegression().fit(xs.reshape(-1, 1), yt)
print(f"기본 설정(C=1.0)  : w = {d.coef_[0, 0]:.2f}")

직선(선형 회귀)과 학습한 S자 곡선을 함께 그려 봅니다(이북 그림 9.1).

In [ ]:
xx = np.linspace(5, 30, 300)
plt.scatter(r_tr["mean radius"], ytr, s=8, alpha=0.4)
plt.plot(xx, lin.coef_[0] * xx + lin.intercept_, label="linear regression")
plt.plot(xx, sigmoid(w / s * xx + (b - w * m / s)), label="logistic regression (numpy GD)")
plt.xlabel("mean radius"); plt.ylabel("P(malignant)"); plt.ylim(-0.6, 1.8); plt.legend()
plt.show()

## 5. 결정 경계
반지름과 질감 두 특징으로 학습합니다. 결정 경계는 $w_1x_1 + w_2x_2 + b = 0$인 직선입니다.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

feats = ["mean radius", "mean texture"]      # 반지름과 질감
two = make_pipeline(StandardScaler(), LogisticRegression(C=np.inf))
two.fit(Xtr[feats], ytr)
(w1, w2), b = two[-1].coef_[0], two[-1].intercept_[0]
print(f"w1 = {w1:.2f}, w2 = {w2:.2f}, b = {b:.2f}")
print(f"테스트 정확도: {two.score(Xte[feats], yte):.1%}")

In [ ]:
gx, gy = np.meshgrid(np.linspace(6, 29, 200), np.linspace(9, 40, 200))
grid = pd.DataFrame({"mean radius": gx.ravel(), "mean texture": gy.ravel()})
pg_ = two.predict_proba(grid)[:, 1].reshape(gx.shape)
cs = plt.contourf(gx, gy, pg_, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.5)
plt.contour(gx, gy, pg_, levels=[0.5], colors="k")
plt.scatter(Xtr["mean radius"], Xtr["mean texture"], c=ytr, cmap="bwr", s=6)
plt.colorbar(cs, label="P(malignant)")
plt.xlabel("mean radius"); plt.ylabel("mean texture")
plt.show()

특징 30개를 모두 쓰고(기본 설정), 임곗값을 바꾸어 봅니다. 4장의 k-최근접 이웃은 테스트 정확도 95.8%, AUC 0.986이었습니다.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

full = make_pipeline(StandardScaler(), LogisticRegression())
full.fit(Xtr, ytr)                           # 특징 30개, 기본 설정
proba = full.predict_proba(Xte)[:, 1]        # 악성일 확률
print(f"테스트 정확도: {full.score(Xte, yte):.1%}, "
      f"AUC: {roc_auc_score(yte, proba):.3f}")
for th in [0.5, 0.3, 0.1]:
    pred = (proba >= th).astype(int)
    tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
    print(f"임곗값 {th}: 놓친 암 {fn}, 잘못된 경보 {fp}")

## 6. 소프트맥스 회귀
부리 길이와 날개 길이로 펭귄 세 종을 분류합니다. `coef_`의 행 하나가 종 하나의 직선입니다.

In [ ]:
import seaborn as sns

pg = sns.load_dataset("penguins").dropna(
    subset=["bill_length_mm", "flipper_length_mm"])
P = pg[["bill_length_mm", "flipper_length_mm"]]
Ptr, Pte, qtr, qte = train_test_split(
    P, pg["species"], test_size=0.25, random_state=42,
    stratify=pg["species"])
soft = make_pipeline(StandardScaler(), LogisticRegression())
soft.fit(Ptr, qtr)
print(soft.classes_)
print(soft[-1].coef_.round(2))
print(soft[-1].intercept_.round(2))
print(f"테스트 정확도: {soft.score(Pte, qte):.1%}")

한 마리의 점수 $z_k$(`decision_function`)로 소프트맥스 $e^{z_k}/(e^{z_1} + e^{z_2} + e^{z_3})$를 직접 계산해 `predict_proba`와 비교합니다.

In [ ]:
one = Pte.iloc[[0]]                          # 테스트 데이터의 첫 펭귄
zk = soft.decision_function(one)[0]          # 종마다의 점수
print("점수:", zk.round(3))
print("직접 계산:", (np.exp(zk) / np.exp(zk).sum()).round(3))
print("predict_proba:", soft.predict_proba(one)[0].round(3))
print("실제 종:", qte.iloc[0])

## 7. 더 알아보기: 서포트 벡터 머신
반달 데이터에서 로지스틱 회귀, 선형 SVM, RBF 커널 SVM을 비교합니다. 서포트 벡터 머신은 표준화를 꼭 함께 씁니다.

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import SVC

Xm, ym = make_moons(n_samples=200, noise=0.2, random_state=0)
Ma, Mb, ma, mb = train_test_split(
    Xm, ym, test_size=0.25, random_state=0, stratify=ym)
for name, clf in [("로지스틱 회귀", LogisticRegression()),
                  ("선형 SVM", SVC(kernel="linear")),
                  ("RBF 커널 SVM", SVC(kernel="rbf"))]:
    model = make_pipeline(StandardScaler(), clf).fit(Ma, ma)
    print(f"{name}: 테스트 정확도 {model.score(Mb, mb):.1%}")

---
## 실습 문제

**문제 1.** 특징 30개 모델(`full`)의 임곗값을 0.5, 0.4, 0.3, 0.2, 0.1, 0.05로 바꾸며 놓친 암, 잘못된 경보, 정밀도, 재현율을 표로 출력하세요. 놓친 암이 1건 이하가 되는 가장 큰 임곗값은 얼마인가요? (원래는 검증 데이터로 정해야 하지만 여기서는 테스트 데이터로 관계만 봅니다.)

In [ ]:
# 문제 1 풀이

**문제 2.** 반지름과 질감 두 특징으로 로지스틱 회귀를 넘파이 경사 하강법으로 학습하세요. 두 특징은 학습 데이터의 평균과 표준편차로 표준화하고, 가중치 $w_j$의 기울기는 $\frac{1}{n}\sum(\hat{y}_i - y_i)x_{ij}$로 계산합니다. 학습률 1.0으로 3,000번 업데이트한 $w_1, w_2, b$를 `two`의 계수와 비교하세요.

In [ ]:
# 문제 2 풀이

**문제 3.** 특징 30개 파이프라인에서 `C`를 0.01, 0.1, 1, 10, 100, `np.inf`로 바꾸어(`max_iter=10000`) 학습하세요. `C`마다 가중치 절댓값의 최댓값, 학습·테스트 정확도, 테스트 로그 손실을 출력하고, 규제가 약해질수록 무엇이 변하는지 설명하세요.

In [ ]:
# 문제 3 풀이

**문제 4.** 펭귄의 수치 특징 네 개(부리 길이, 부리 깊이, 날개 길이, 몸무게)로 소프트맥스 회귀를 학습하고, 같은 나눔에서 k-최근접 이웃($k=5$)과 테스트 정확도를 비교하세요. 두 모델 모두 표준화 파이프라인을 쓰고, 네 특징에 빈 값이 있는 펭귄은 뺍니다.

In [ ]:
# 문제 4 풀이

**문제 5.** 반달 데이터에서 RBF 커널 SVM의 `C`를 0.1, 1, 10, 100, 1000으로 바꾸어 학습·테스트 정확도와 서포트 벡터의 수(`n_support_`의 합)를 비교하세요.

In [ ]:
# 문제 5 풀이